# embedding EDA

In this notebook, we figure out the huggingface embedding dataset.
We need to map these to the page ids.
We also need to get one embedding per page.
Then we can also explore what it'll take in order to index this into faiss (with a 1-10% sample).

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession


def get_spark(cores=8, memory="60g"):
    spark = (
        SparkSession.builder.master(f"local[{cores}]")
        .config("spark.driver.memory", memory)
        .getOrCreate()
    )
    return spark


root = Path("~/scratch/trec-tot-2025/data").expanduser()
emb_root = root / "wikipedia-2024-06-bge-m3/data/en"
page_root = root / "enwiki/parquet/page"

spark = get_spark()
emb = spark.read.parquet(emb_root.as_posix())
page = spark.read.parquet(page_root.as_posix())
emb.printSchema()
page.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/16 02:40:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- embedding: array (nullable = true)
 |    |-- element: float (containsNull = true)

root
 |-- page_id: integer (nullable = true)
 |-- page_namespace: integer (nullable = true)
 |-- page_title: string (nullable = true)
 |-- page_is_redirect: string (nullable = true)
 |-- page_is_new: string (nullable = true)
 |-- page_random: string (nullable = true)
 |-- page_touched: string (nullable = true)
 |-- page_links_updated: string (nullable = true)
 |-- page_latest: string (nullable = true)
 |-- page_len: string (nullable = true)
 |-- page_content_model: string (nullable = true)
 |-- page_lang: string (nullable = true)



In [3]:
emb.show(3, vertical=True, truncate=80)

-RECORD 0-------------------------------------------------------------------------------------
 id        | 26348708_2                                                                       
 url       | https://en.wikipedia.org/wiki/Perry%20Municipal%20Airport%20%28Oklahoma%29       
 title     | Perry Municipal Airport (Oklahoma)                                               
 text      | Flying training was performed with Fairchild PT-19s as the primary trainer. A... 
 embedding | [-0.027695153, 0.0103476215, -0.039931383, -0.023554558, 0.009153579, -0.0545... 
-RECORD 1-------------------------------------------------------------------------------------
 id        | 26348708_3                                                                       
 url       | https://en.wikipedia.org/wiki/Perry%20Municipal%20Airport%20%28Oklahoma%29       
 title     | Perry Municipal Airport (Oklahoma)                                               
 text      | Perry Municipal Airport covers an are

In [7]:
emb.count()

47018430

In [5]:
from pyspark.sql import functions as F

emb_id = emb.select(F.split("id", "_")[0].alias("id")).distinct()
emb_id.count()

6614232

In [ ]:
# now join with page to see how many pages have embeddings
page_id = page.select(F.col("page_id").alias("id")).cache()
joined = page_id.distinct().join(emb_id, on="id", how="inner")

page_id.count(), joined.count()

(63415160, 6596109)

In [ ]:
page_id.where('page_namespace=0 and page_is_redirect="0"').count()

7016300

In [ ]:
6596109 / 7016300

0.9401121673816684

In [ ]:
import numpy as np


@F.udf(returnType="array<float>")
def avg_vector(vectors: list) -> list:
    return np.mean(np.array(vectors), axis=0).tolist()


# let's compute a few averaged embeddings to see if we can do this properly
emb_avg = (
    emb.withColumn("page_id", F.split("id", "_")[0])
    .where(F.crc32("id") % 100 == 0)
    .groupBy("id")
    .agg(F.collect_list("embedding").alias("embeddings"))
    .select("id", avg_vector("embeddings").alias("embedding"))
)